In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dev.configs;

CREATE OR REPLACE FUNCTION dev.configs.mask_phone(p STRING)
RETURNS STRING
RETURN CASE WHEN p IS NULL THEN NULL ELSE concat('***-***-', right(p, 4)) END;

CREATE POLICY mask_phone_policy
ON CATALOG dev
COMMENT 'Mask any column tagged phone across the dev catalog'
COLUMN MASK dev.configs.mask_phone
TO `account users` EXCEPT `workspace-admins`
FOR TABLES
MATCH COLUMNS has_tag('phone') AS phone
ON COLUMN phone;



In [0]:
%sql
-- Row filter: sales persons see only their own customers
CREATE TABLE IF NOT EXISTS dev.configs.salesperson_user_map (
  sales_person STRING COMMENT 'Value as stored in SalesPerson column, e.g. adventure-works\\jillian0',
  user_email   STRING COMMENT 'Databricks user email (current_user())'
);

-- Example mapping row (replace with real users):
INSERT INTO dev.configs.salesperson_user_map VALUES ('adventure-works\\jillian0', 'rajkumar@r7755630gmail.onmicrosoft.com');

CREATE OR REPLACE FUNCTION dev.configs.filter_by_salesperson(sp STRING)
RETURNS BOOLEAN
RETURN EXISTS (
  SELECT 1 FROM dev.configs.salesperson_user_map m
  WHERE m.sales_person = sp AND lower(m.user_email) = lower(current_user())
);

ALTER TABLE dev.`1_bronze`.customer ALTER COLUMN SalesPerson SET TAGS ('sales_owner');

CREATE POLICY salesperson_row_filter
ON CATALOG dev
COMMENT 'Show only rows owned by the current sales person on any column tagged sales_owner'
ROW FILTER dev.configs.filter_by_salesperson
TO `account users` EXCEPT `workspace-admins`
FOR TABLES
MATCH COLUMNS has_tag('sales_owner') AS sp
USING COLUMNS (sp);